In [5]:
import networkx as nx
import numpy as np
import pandas as pd
import time
import igraph as ig
import leidenalg
import xlsxwriter

start_time = time.time()

# --- Step 1: Build Bipartite Graph ---
G = nx.Graph()
with open(r"C:\Users\Dell\sarada\Sarada_paper1, conferences\B.Tech_Akshya\Restarent Recommendations\Datasets for RS\Datasets for RS\Restaurant Recommendations\restaurantupdate.txt", 'r') as f:
    for line in f:
        u1, v1, r = line.strip().split('\t')
        u = "A" + u1
        v = "B" + v1
        G.add_node(u, bipartite=0)
        G.add_node(v, bipartite=1)
        G.add_edge(u, v)

# --- Step 2: Ratings Matrix ---
ratings_df = pd.read_csv(r"C:\Users\Dell\sarada\Sarada_paper1, conferences\B.Tech_Akshya\Restarent Recommendations\Datasets for RS\Datasets for RS\Restaurant Recommendations\restaurantupdate.txt",
                         delimiter='\t', header=None, names=['user', 'item', 'rating'])
ratings_df['user'] = 'A' + ratings_df['user'].astype(str)
ratings_df['item'] = 'B' + ratings_df['item'].astype(str)
matrix = ratings_df.pivot(index='user', columns='item', values='rating').fillna(0)
matrix_R = matrix.to_numpy()

# --- Utility: Matrix Factorization ---
def matrix_factorization(R, P, Q, K, steps=100, alpha=0.005, beta=0.02):
    Q = Q.T
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    eij = R[i][j] - np.dot(P[i, :], Q[:, j])
                    for k in range(K):
                        P[i][k] += alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] += alpha * (2 * eij * P[i][k] - beta * Q[k][j])
        e = np.sum((R - np.dot(P, Q)) ** 2)
        if e < 0.001:
            break
    return P, Q.T

# --- Leiden Community Detection ---
def leiden_algorithm(G_nx, resolution):
    mapping = {node: idx for idx, node in enumerate(G_nx.nodes())}
    reverse_mapping = {idx: node for node, idx in mapping.items()}
    edges = [(mapping[u], mapping[v]) for u, v in G_nx.edges()]

    G_ig = ig.Graph()
    G_ig.add_vertices(len(mapping))
    if edges:
        G_ig.add_edges(edges)

    partition = leidenalg.find_partition(G_ig, leidenalg.CPMVertexPartition,
                                         resolution_parameter=resolution)

    partitions = [[reverse_mapping[node] for node in community] for community in partition]
    return partitions

# --- Modified Jaccard Similarity ---
def modified_jaccard_similarity(R):
    num_users, num_items = R.shape
    sim = np.zeros((num_items, num_items))
    for i in range(num_items):
        users_i = set(np.where(R[:, i] > 0)[0])
        for j in range(num_items):
            users_j = set(np.where(R[:, j] > 0)[0])
            intersection = len(users_i & users_j)
            denom = len(users_i) * len(users_j)
            sim[i, j] = intersection / denom if denom > 0 else 0.0
    return sim

# --- Parameters ---
resolutions = [0.005, 0.025, 0.05, 0.075, 0.1, 0.125, 0.154, 0.175, 0.25]
custom_indices = [1, 50, 100, 150, 200, 250, 300, 350, 400]
weights = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]

mae_records = []

# --- Main Loop ---
for comm_idx, res in zip(custom_indices, resolutions):
    print(f"\nProcessing resolution {res} (community {comm_idx})")
    communities = leiden_algorithm(G, resolution=res)

    selected_part = None
    for part in communities:
        part_users = [u for u in part if u in matrix.index]
        part_items = [i for i in part if i in matrix.columns]
        if part_users and part_items:
            selected_part = matrix.loc[part_users, part_items].to_numpy()
            community_size = len(part)
            break

    if selected_part is None:
        continue

    R = selected_part

    K = 20
    P = np.random.rand(R.shape[0], K)
    Q = np.random.rand(R.shape[1], K)
    nP, nQ = matrix_factorization(R, P, Q, K, steps=50)
    nR = np.dot(nP, nQ.T)

    content_sim = modified_jaccard_similarity(R)
    content_based_matrix = np.dot(R, content_sim)

    mae_list = []
    for w in weights:
        pred_matrix_final = w * content_based_matrix + (1 - w) * nR
        MAE_final = np.sum(np.abs(pred_matrix_final - R)) / R.size
        mae_list.append(MAE_final)

    record = {"Resolution": res, "Community Index": comm_idx, "Community Size": community_size}
    for i, mae in enumerate(mae_list):
        record[f"MAE_w{i*0.1:.1f}"] = round(mae, 4)

    mae_records.append(record)

# --- Save MAE to Excel ---
mae_df = pd.DataFrame(mae_records)
output_path = r"C:\Users\Dell\sarada\Sarada_paper1, conferences\B.Tech_Akshya\Restarent Recommendations.xlsx"
mae_df.to_excel(output_path, index=False)

print("MAE results saved successfully!")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")



Processing resolution 0.005 (community 1)

Processing resolution 0.025 (community 50)

Processing resolution 0.05 (community 100)

Processing resolution 0.075 (community 150)

Processing resolution 0.1 (community 200)

Processing resolution 0.125 (community 250)

Processing resolution 0.154 (community 300)

Processing resolution 0.175 (community 350)

Processing resolution 0.25 (community 400)
MAE results saved successfully!
Total execution time: 4.47 seconds


In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import time
import igraph as ig
import leidenalg
import xlsxwriter

start_time = time.time()

# --- Step 1: Build Bipartite Graph ---
G = nx.Graph()
with open(r"C:\Users\Dell\sarada\ml-100k .txt", 'r') as f:
    for line in f:
        u1, v1, r = line.strip().split('\t')
        u = "A" + u1
        v = "B" + v1
        G.add_node(u, bipartite=0)
        G.add_node(v, bipartite=1)
        G.add_edge(u, v)

# --- Step 2: Ratings Matrix ---
ratings_df = pd.read_csv(r"C:\Users\Dell\sarada\ml-100k .txt", delimiter='\t', header=None,
                         names=['user', 'item', 'rating'])
ratings_df['user'] = 'A' + ratings_df['user'].astype(str)
ratings_df['item'] = 'B' + ratings_df['item'].astype(str)
matrix = ratings_df.pivot(index='user', columns='item', values='rating').fillna(0)
matrix_R = matrix.to_numpy()

# --- Utility: Matrix Factorization ---
def matrix_factorization(R, P, Q, K, steps=100, alpha=0.005, beta=0.02):
    Q = Q.T
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    eij = R[i][j] - np.dot(P[i, :], Q[:, j])
                    for k in range(K):
                        P[i][k] += alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] += alpha * (2 * eij * P[i][k] - beta * Q[k][j])
        e = np.sum((R - np.dot(P, Q)) ** 2)
        if e < 0.001:
            break
    return P, Q.T

# --- Leiden Community Detection ---
def leiden_algorithm(G_nx, resolution):
    mapping = {node: idx for idx, node in enumerate(G_nx.nodes())}
    reverse_mapping = {idx: node for node, idx in mapping.items()}
    edges = [(mapping[u], mapping[v]) for u, v in G_nx.edges()]

    G_ig = ig.Graph()
    G_ig.add_vertices(len(mapping))
    if edges:
        G_ig.add_edges(edges)

    partition = leidenalg.find_partition(
        G_ig,
        leidenalg.CPMVertexPartition,
        resolution_parameter=resolution
    )

    partitions = [
        [reverse_mapping[node] for node in community]
        for community in partition
    ]
    return partitions

# --- Modified Jaccard Similarity ---
def modified_jaccard_similarity(R):
    """Computes Modified Jaccard Similarity for items."""
    num_users, num_items = R.shape
    sim = np.zeros((num_items, num_items))
    for i in range(num_items):
        users_i = set(np.where(R[:, i] > 0)[0])
        for j in range(num_items):
            users_j = set(np.where(R[:, j] > 0)[0])
            intersection = len(users_i & users_j)
            denom = len(users_i) * len(users_j)
            sim[i, j] = intersection / denom if denom > 0 else 0.0
    return sim

# --- Load Genre Data ---
genre_df = pd.read_csv(r"C:\Users\Dell\sarada\u.item", sep='|', encoding='latin-1', header=None, usecols=range(24))
columns = ["movie_id", "movie_title", "release_date", "video_release_date", "IMDb_URL",
           "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
           "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
           "Romance", "Sci-Fi", "Thriller", "War", "Western"]
genre_df.columns = columns
genre_df['item'] = 'B' + genre_df['movie_id'].astype(str)
selected_columns = ["unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
                    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
                    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]
genre_data = genre_df[["item"] + selected_columns]
genre_data.set_index('item', inplace=True)
genre_data = genre_data.fillna(0)

# --- Step 3: Resolutions and Parameters ---
resolutions = [0.005, 0.025, 0.05, 0.075, 0.1, 0.125, 0.154, 0.175, 0.25]
custom_indices = [1, 50, 100, 150, 200, 250, 300, 350, 400]
weights = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]

rmse_records = []

# --- Step 4: Main Loop ---
for comm_idx, res in zip(custom_indices, resolutions):
    print(f"\n=== Processing resolution {res} (index {comm_idx}) ===")
    communities = leiden_algorithm(G, resolution=res)

    selected_part = None
    for part in communities:
        part_users = [u for u in part if u in matrix.index]
        part_items = [i for i in part if i in matrix.columns]
        if not part_users or not part_items:
            continue

        aligned_items = [item for item in part_items if item in genre_data.index]
        if not aligned_items:
            continue

        part_matrix = matrix.loc[part_users, aligned_items]
        part_genre = genre_data.loc[aligned_items]

        R = part_matrix.to_numpy()
        genre = part_genre.to_numpy()

        if R.size == 0 or genre.size == 0:
            continue

        selected_part = (R, genre)
        community_size = len(part)
        break

    if selected_part is None:
        print(f"No valid community found for resolution {res}")
        continue

    R, genre = selected_part

    # --- Matrix Factorization (Collaborative Filtering) ---
    K = 20
    P = np.random.uniform(0.0, 1.0, size=(R.shape[0], K))
    Q = np.random.uniform(0.0, 1.0, size=(R.shape[1], K))
    nP, nQ = matrix_factorization(R, P, Q, K, steps=50, alpha=0.005, beta=0.02)
    nR = np.dot(nP, nQ.T)

    # --- Content-Based Filtering (Modified Jaccard) ---
    content_sim = modified_jaccard_similarity(R)
    content_based_matrix = np.dot(R, content_sim)

    # --- Normalize sizes if needed ---
    if content_based_matrix.shape != nR.shape:
        min_rows = min(content_based_matrix.shape[0], nR.shape[0])
        min_cols = min(content_based_matrix.shape[1], nR.shape[1])
        content_based_matrix = content_based_matrix[:min_rows, :min_cols]
        nR = nR[:min_rows, :min_cols]
        R = R[:min_rows, :min_cols]

    # --- RMSE Computation ---
    rmse_list = []
    for w in weights:
        pred_matrix_final = w * content_based_matrix + (1 - w) * nR
        RMSE_final = np.sqrt(np.sum((pred_matrix_final - R) ** 2) / R.size)
        rmse_list.append(RMSE_final)

    record = {
        "Resolution": res,
        "Community Index": comm_idx,
        "Community Size": community_size
    }
    for i, rmse in enumerate(rmse_list):
        record[f"RMSE_w{i*0.1:.1f}"] = round(rmse, 4)
    rmse_records.append(record)

# --- Step 5: Save to Excel ---
rmse_df = pd.DataFrame(rmse_records)
output_path = r"C:\Users\Dell\sarada\B.Tech_Akshya\RMSE_K=20.xlsx"
rmse_df.to_excel(output_path, index=False)

print("\n Only RMSE results saved to Excel successfully!")
print(f"Saved to: {output_path}")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")


In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import time
import igraph as ig
import leidenalg
import xlsxwriter

start_time = time.time()

# --- Step 1: Build Bipartite Graph ---
G = nx.Graph()
with open(r"C:\Users\Dell\sarada\ml-100k .txt", 'r') as f:
    for line in f:
        u1, v1, r = line.strip().split('\t')
        u = "A" + u1
        v = "B" + v1
        G.add_node(u, bipartite=0)
        G.add_node(v, bipartite=1)
        G.add_edge(u, v)

# --- Step 2: Ratings Matrix ---
ratings_df = pd.read_csv(r"C:\Users\Dell\sarada\ml-100k .txt", delimiter='\t', header=None,
                         names=['user', 'item', 'rating'])
ratings_df['user'] = 'A' + ratings_df['user'].astype(str)
ratings_df['item'] = 'B' + ratings_df['item'].astype(str)
matrix = ratings_df.pivot(index='user', columns='item', values='rating').fillna(0)
matrix_R = matrix.to_numpy()

# --- Utility: Matrix Factorization ---
def matrix_factorization(R, P, Q, K, steps=100, alpha=0.005, beta=0.02):
    Q = Q.T
    for step in range(steps):
        for i in range(len(R)):
            for j in range(len(R[i])):
                if R[i][j] > 0:
                    eij = R[i][j] - np.dot(P[i, :], Q[:, j])
                    for k in range(K):
                        P[i][k] += alpha * (2 * eij * Q[k][j] - beta * P[i][k])
                        Q[k][j] += alpha * (2 * eij * P[i][k] - beta * Q[k][j])
        e = np.sum((R - np.dot(P, Q)) ** 2)
        if e < 0.001:
            break
    return P, Q.T

# --- Leiden Community Detection ---
def leiden_algorithm(G_nx, resolution):
    mapping = {node: idx for idx, node in enumerate(G_nx.nodes())}
    reverse_mapping = {idx: node for node, idx in mapping.items()}
    edges = [(mapping[u], mapping[v]) for u, v in G_nx.edges()]

    G_ig = ig.Graph()
    G_ig.add_vertices(len(mapping))
    if edges:
        G_ig.add_edges(edges)

    partition = leidenalg.find_partition(
        G_ig,
        leidenalg.CPMVertexPartition,
        resolution_parameter=resolution
    )

    partitions = [
        [reverse_mapping[node] for node in community]
        for community in partition
    ]
    return partitions

# --- Modified Jaccard Similarity ---
def modified_jaccard_similarity(R):
    """Computes Modified Jaccard Similarity for items."""
    num_users, num_items = R.shape
    sim = np.zeros((num_items, num_items))
    for i in range(num_items):
        users_i = set(np.where(R[:, i] > 0)[0])
        for j in range(num_items):
            users_j = set(np.where(R[:, j] > 0)[0])
            intersection = len(users_i & users_j)
            denom = len(users_i) * len(users_j)
            sim[i, j] = intersection / denom if denom > 0 else 0.0
    return sim

# --- Load Genre Data ---
genre_df = pd.read_csv(r"C:\Users\Dell\sarada\u.item", sep='|', encoding='latin-1', header=None, usecols=range(24))
columns = ["movie_id", "movie_title", "release_date", "video_release_date", "IMDb_URL",
           "unknown", "Action", "Adventure", "Animation", "Children's", "Comedy", "Crime",
           "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror", "Musical", "Mystery",
           "Romance", "Sci-Fi", "Thriller", "War", "Western"]
genre_df.columns = columns
genre_df['item'] = 'B' + genre_df['movie_id'].astype(str)
selected_columns = ["unknown", "Action", "Adventure", "Animation", "Children's", "Comedy",
                    "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir", "Horror",
                    "Musical", "Mystery", "Romance", "Sci-Fi", "Thriller", "War", "Western"]
genre_data = genre_df[["item"] + selected_columns]
genre_data.set_index('item', inplace=True)
genre_data = genre_data.fillna(0)

# --- Step 3: Resolutions and Parameters ---
resolutions = [0.005, 0.025, 0.05, 0.075, 0.1, 0.125, 0.154, 0.175, 0.25]
custom_indices = [1, 50, 100, 150, 200, 250, 300, 350, 400]
weights = [round(x, 1) for x in np.arange(0, 1.01, 0.1)]

rmse_records = []

# --- Step 4: Main Loop ---
for comm_idx, res in zip(custom_indices, resolutions):
    print(f"\n=== Processing resolution {res} (index {comm_idx}) ===")
    communities = leiden_algorithm(G, resolution=res)

    selected_part = None
    for part in communities:
        part_users = [u for u in part if u in matrix.index]
        part_items = [i for i in part if i in matrix.columns]
        if not part_users or not part_items:
            continue

        aligned_items = [item for item in part_items if item in genre_data.index]
        if not aligned_items:
            continue

        part_matrix = matrix.loc[part_users, aligned_items]
        part_genre = genre_data.loc[aligned_items]

        R = part_matrix.to_numpy()
        genre = part_genre.to_numpy()

        if R.size == 0 or genre.size == 0:
            continue

        selected_part = (R, genre)
        community_size = len(part)
        break

    if selected_part is None:
        print(f"No valid community found for resolution {res}")
        continue

    R, genre = selected_part

    # --- Matrix Factorization (Collaborative Filtering) ---
    K = 20
    P = np.random.uniform(0.0, 1.0, size=(R.shape[0], K))
    Q = np.random.uniform(0.0, 1.0, size=(R.shape[1], K))
    nP, nQ = matrix_factorization(R, P, Q, K, steps=50, alpha=0.005, beta=0.02)
    nR = np.dot(nP, nQ.T)

    # --- Content-Based Filtering (Modified Jaccard) ---
    content_sim = modified_jaccard_similarity(R)
    content_based_matrix = np.dot(R, content_sim)

    # --- Normalize sizes if needed ---
    if content_based_matrix.shape != nR.shape:
        min_rows = min(content_based_matrix.shape[0], nR.shape[0])
        min_cols = min(content_based_matrix.shape[1], nR.shape[1])
        content_based_matrix = content_based_matrix[:min_rows, :min_cols]
        nR = nR[:min_rows, :min_cols]
        R = R[:min_rows, :min_cols]

    # --- RMSE Computation ---
    rmse_list = []
    for w in weights:
        pred_matrix_final = w * content_based_matrix + (1 - w) * nR
        RMSE_final = np.sqrt(np.sum((pred_matrix_final - R) ** 2) / R.size)
        rmse_list.append(RMSE_final)

    record = {
        "Resolution": res,
        "Community Index": comm_idx,
        "Community Size": community_size
    }
    for i, rmse in enumerate(rmse_list):
        record[f"RMSE_w{i*0.1:.1f}"] = round(rmse, 4)
    rmse_records.append(record)

# --- Step 5: Save to Excel ---
rmse_df = pd.DataFrame(rmse_records)
output_path = r"C:\Users\Dell\sarada\B.Tech_Akshya\K=30.xlsx"
rmse_df.to_excel(output_path, index=False)

print("\n Only RMSE results saved to Excel successfully!")
print(f"Saved to: {output_path}")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")
